# Ensure SC Auditor Service Principal

Creates (or finds) a least-privilege service principal dedicated to the SC Auditor screen capture audit tool. The SPN is scoped to a single schema and secret scope — it cannot be reused for unrelated workloads.

This notebook runs as the **first task** in the `sc_auditor_uc_setup` job. It outputs the SPN's `application_id` as a **task value** so the downstream DDL task can grant it table-level permissions.

**Secret scope keys auto-provisioned by this notebook:**

| Key | Name Variable | Source | Purpose |
| --- | --- | --- | --- |
| Client ID | `client_id_dbs_key` job param | SPN `application_id` | OAuth M2M client identifier |
| Workspace URL | `workspace_url` (fixed) | Derived from config | Databricks workspace URL for SDK/API calls |

**Secret scope key requiring manual admin provisioning:**

| Key | Name Variable | Source | Purpose |
| --- | --- | --- | --- |
| Client secret | `client_secret_dbs_key` job param | Admin-generated | OAuth M2M client secret |

Key names are **schema-qualified** in dev and hls_fde targets (e.g. `client_id_sc_auditor`,
`client_secret_sc_auditor`) so that multiple schemas can share a single secret scope
without key collisions. The actual key names are passed via job parameters.

> **Admin step required:** After the first run of this job, an admin must generate an OAuth secret for the SPN and populate the client secret key (named by `client_secret_dbs_key`) in the secret scope. This can be done via:
> * **Workspace UI:** Settings → Identity and access → Service principals → select the SPN → Secrets → Generate secret
> * **Account Console:** User management → Service principals → Credentials & secrets
> * **Databricks CLI:** `databricks account service-principal-secrets create <workspace_sp_id>`
> * **External keystore:** Sync credentials from Azure Key Vault, AWS Secrets Manager, or HashiCorp Vault
>
> The client ID is stored automatically under the key named by `client_id_dbs_key`. Only the client secret needs manual provisioning.

**Permissions handled here:**
* Secret scope `READ` ACL — so the Databricks App can retrieve all secrets at runtime

**Permissions handled by the DDL task (downstream):**
* `USE CATALOG`, `USE SCHEMA`, `MODIFY`, `SELECT` on the target tables
* `READ FILES`, `WRITE FILES` on the UC Volumes

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
# Get job parameters
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
secret_scope_name = dbutils.widgets.get("secret_scope_name")
client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
client_secret_dbs_key = dbutils.widgets.get("client_secret_dbs_key")

print(f"Catalog:            {catalog_use}")
print(f"Schema:             {schema_use}")
print(f"Secret scope:       {secret_scope_name}")
print(f"Client ID key:      {client_id_dbs_key}")
print(f"Client secret key:  {client_secret_dbs_key}")

In [ ]:
# Find or create the service principal
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Naming convention: sc-auditor-{schema} — ties the SPN to this schema
spn_display_name = f"sc-auditor-{schema_use}"
print(f"Target SPN display name: {spn_display_name}")

# Search for an existing SPN with this name
existing_spns = list(
    w.service_principals.list(filter=f'displayName eq "{spn_display_name}"')
)

if existing_spns:
    spn = existing_spns[0]
    is_new_spn = False
    print(f"Found existing SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")
    print(f"  active: {spn.active}")
else:
    spn = w.service_principals.create(
        display_name=spn_display_name,
        active=True
    )
    is_new_spn = True
    print(f"Created new SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")

spn_application_id = spn.application_id

In [ ]:
# ---------------------------------------------------------------------------
# Store derived values and the SPN's client_id in the secret scope, then
# verify that the admin-provisioned client_secret is present.
#
# Auto-provisioned (key names from job params):
#   {client_id_dbs_key}      — SPN application_id (OAuth M2M client identifier)
#   workspace_url            — Databricks workspace URL
# These are refreshed on every run.
#
# Admin-provisioned:
#   {client_secret_dbs_key} — OAuth M2M client secret
#   Must be created via workspace UI, account console, CLI, or external keystore.
# ---------------------------------------------------------------------------

workspace_url = w.config.host.rstrip("/")
print(f"Workspace URL: {workspace_url}")

# ---- Store auto-provisioned values in the secret scope --------------------
try:
    secrets_to_store = {
        client_id_dbs_key: spn_application_id,
        "workspace_url":   workspace_url,
    }
    for key, value in secrets_to_store.items():
        w.secrets.put_secret(scope=secret_scope_name, key=key, string_value=value)
    print(f"\nStored in scope '{secret_scope_name}':")
    print(f"  {client_id_dbs_key:<22s} = {spn_application_id}")
    print(f"  {'workspace_url':<22s} = {workspace_url}")
except Exception as e:
    print(f"\nWarning: could not store secrets (scope may not exist yet): {e}")

# ---- Check for admin-provisioned client_secret ----------------------------
try:
    existing_keys = {s.key for s in w.secrets.list_secrets(scope=secret_scope_name)}
    credentials_provisioned = client_secret_dbs_key in existing_keys
    print(f"\nSecret scope keys: {sorted(existing_keys)}")
    if credentials_provisioned:
        print(f"OAuth client secret ({client_secret_dbs_key}): PRESENT")
    else:
        print(f"OAuth client secret ({client_secret_dbs_key}): MISSING")
        print()
        print("=" * 65)
        print("ACTION REQUIRED: An admin must provision the client_secret.")
        print("=" * 65)
        print()
        print(f"SPN display name:   {spn_display_name}")
        print(f"SPN application_id: {spn_application_id} (stored as '{client_id_dbs_key}')")
        print(f"SPN workspace ID:   {spn.id}")
        print()
        print("Option 1 — Workspace UI:")
        print("  Settings > Identity and access > Service principals")
        print(f"  > Select '{spn_display_name}' > Secrets > Generate secret")
        print()
        print("Option 2 — Databricks CLI (requires account admin):")
        print(f"  databricks account service-principal-secrets create {spn.id}")
        print()
        print("Then store the secret:")
        print(f'  databricks secrets put-secret --scope {secret_scope_name} --key {client_secret_dbs_key} --string-value "<secret>"')
except Exception:
    credentials_provisioned = False
    print(f"\nCould not check credentials (scope '{secret_scope_name}' may not exist yet).")

In [ ]:
# Grant the SPN READ access to the secret scope so the Databricks App
# can retrieve client_id, client_secret, and workspace_url at runtime.
# put_acl is idempotent — safe to re-run.
from databricks.sdk.service.workspace import AclPermission

try:
    w.secrets.put_acl(
        scope=secret_scope_name,
        principal=spn_application_id,
        permission=AclPermission.READ
    )
    print(f"Granted READ on scope '{secret_scope_name}' to {spn_application_id}")
except Exception as e:
    # If the scope doesn't exist yet (first deploy before bundle creates it),
    # log the error but don't fail — the scope will be created by the bundle.
    print(f"Warning: Could not set secret scope ACL: {e}")
    print("The secret scope may not exist yet. Re-run after bundle deploy.")

In [ ]:
# Pass the SPN application_id to the next task in the job.
# The DDL task reads this via:
#   {{tasks.ensure_service_principal.values.spn_application_id}}
dbutils.jobs.taskValues.set(key="spn_application_id", value=spn_application_id)
print(f"Task value set: spn_application_id = {spn_application_id}")

In [ ]:
# Summary
print("=" * 65)
print("Service Principal Summary")
print("=" * 65)
print(f"  Display name:      {spn_display_name}")
print(f"  Application ID:    {spn_application_id}")
print(f"  Newly created:     {is_new_spn}")
print(f"  Secret scope:      {secret_scope_name}")
print(f"    READ ACL:        granted")
print(f"    {client_id_dbs_key + ':':<22s} stored")
print(f"    {'workspace_url:':<22s} stored")
print(f"    {client_secret_dbs_key + ':':<22s} {'PRESENT' if credentials_provisioned else 'MISSING (admin action required)'}")
print(f"  Target schema:     {catalog_use}.{schema_use}")
print()
if not credentials_provisioned:
    print(f"*** The {client_secret_dbs_key} must be provisioned by an admin  ***")
    print("*** before the Databricks App can authenticate.                 ***")
    print()
print("UC grants (USE CATALOG, USE SCHEMA, MODIFY, SELECT)")
print("and Volume permissions will be applied by the downstream DDL task.")
print("=" * 65)